# Faruq-v3 AF2FS + IGEM Frozen-Parent Paired Confirmation — Kaggle

Audited 2026-08-24. Enam development-validation run: `AF2IGEM0/1 × seed 42/123/2026`.
Setiap arm memakai **AF2FS best checkpoint seed-matched yang dibekukan penuh**; hanya residual IGEM yang trainable.
Test tetap locked.

Required Kaggle inputs:
- `faruq-v3-experiment-core-v1`;
- `af2-ffab2-all-seeds-all-stages-state.zip`;
- opsional `af2-igem-parent-confirmation-state.zip` untuk melanjutkan arm yang sudah selesai.

GPU + Internet ON. Notebook fail-closed sebelum training.


In [ ]:
from pathlib import Path
import importlib,json,os,shutil,subprocess,sys,time,zipfile

INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir():
    raise RuntimeError('Kaggle-only notebook')
os.chdir(WORK)

print('INDEXING /kaggle/input ONCE ...',flush=True)
t0=time.time(); INPUT_INDEX={}; file_count=0
for root,dirs,files in os.walk(INPUT):
    for filename in files:
        p=Path(root)/filename
        INPUT_INDEX.setdefault(filename,[]).append(p)
        file_count+=1
print(f'INPUT INDEX READY: {file_count} files in {time.time()-t0:.2f}s',flush=True)

def input_matches(name):
    return sorted(INPUT_INDEX.get(name,[]))

def input_one(name):
    matches=input_matches(name)
    if len(matches)!=1:
        raise FileNotFoundError(f'Harus tepat satu {name}; ditemukan {matches}')
    return matches[0]

for name in (
    'af2_spectral_kaggle_manifest.json',
    'faruq-development-v3-grouped.tar.bin',
    'D0_seed42_best.pt','D0_seed123_best.pt','D0_seed2026_best.pt',
):
    print('CORE INPUT OK:',name,'->',input_one(name),flush=True)

PRIOR_STATE_NAME='af2-ffab2-all-seeds-all-stages-state.zip'
prior_state=input_one(PRIOR_STATE_NAME)
PRIOR=WORK/'af2fs-igem-parent-source'
if PRIOR.exists():
    shutil.rmtree(PRIOR)
PRIOR.mkdir(parents=True)
with zipfile.ZipFile(prior_state,'r') as z:
    members=[]
    wanted={f'AF2FS_seed{s}_result.json' for s in (42,123,2026)}
    for info in z.infolist():
        base=Path(info.filename).name
        if base in wanted or base=='best.pt':
            members.append(info)
    found_results={Path(info.filename).name for info in members if Path(info.filename).name in wanted}
    if found_results!=wanted:
        raise RuntimeError(f'Prior state tidak memuat semua AF2FS result: {found_results}')
    if not any(Path(info.filename).name=='best.pt' for info in members):
        raise RuntimeError('Prior state tidak memuat AF2FS best.pt')
    for info in members:
        z.extract(info,PRIOR)
print('AF2FS SOURCE RESTORED:',len(members),'members',flush=True)
print('CELL 1 DONE')


In [ ]:
import torch

BRANCH='codex/af2-igem-parent-confirmation'
REPO=WORK/'coffee-bean-detection'

# Critical rerun safety: never delete the directory that is the kernel CWD.
os.chdir(WORK)
if REPO.exists():
    shutil.rmtree(REPO)

for attempt in range(1,4):
    r=subprocess.run(
        ['git','clone','--depth','1','--branch',BRANCH,
         'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],
        cwd=WORK,
    )
    if r.returncode==0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt==3:
        raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)

subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)

for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'):
        sys.modules.pop(name,None)
if str(REPO/'src') not in sys.path:
    sys.path.insert(0,str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)

COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('BRANCH:',BRANCH)
print('COMMIT:',COMMIT)
print('ULTRALYTICS:',__import__('ultralytics').__version__)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available():
    raise RuntimeError('Aktifkan Kaggle GPU sebelum Run All')

tests=[
    'tests/test_af2_parent_residual.py',
    'tests/test_af2_parent_residual_audit_tolerance.py',
    'tests/test_af2_igem_parent_confirmation.py',
]
preflight=subprocess.run(
    [sys.executable,'-m','pytest','-q',*tests],
    cwd=REPO,text=True,capture_output=True,
)
print(preflight.stdout,flush=True)
if preflight.returncode:
    print(preflight.stderr,flush=True)
    raise RuntimeError(f'PRETRAIN CONTRACT TESTS FAILED rc={preflight.returncode}')
print('ALL IGEM PRETRAIN CONTRACT TESTS PASS')


In [ ]:
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
from coffee_detector.af2_spectral.audit import sha256

DATA,ARTIFACTS,CORE=prepare_af2_spectral_kaggle_input(INPUT,WORK)
if CORE.get('decision')!='PASS' or CORE.get('test_images_accessed') is not False:
    raise RuntimeError('Core contract gagal')
if (DATA/'test').exists():
    raise RuntimeError('TEST TEREXPOSE — STOP')

GROUPED=DATA/'faruq_grouped_summary.json'
SEEDS=(42,123,2026)

def find_one(root,filename):
    matches=sorted(root.rglob(filename))
    if len(matches)!=1:
        raise FileNotFoundError(f'Harus tepat satu {filename}; ditemukan {matches}')
    return matches[0].resolve()

PARENT_RESULTS={s:find_one(PRIOR,f'AF2FS_seed{s}_result.json') for s in SEEDS}
best_files=sorted(PRIOR.rglob('best.pt'))

def resolve_parent(seed):
    payload=json.loads(PARENT_RESULTS[seed].read_text(encoding='utf-8'))
    if payload.get('arm')!='AF2FS' or int(payload.get('seed',-1))!=seed:
        raise RuntimeError(f'AF2FS result seed {seed} invalid')
    expected=payload['checkpoint_sha256']
    matches=[p.resolve() for p in best_files if sha256(p)==expected]
    if not matches:
        raise FileNotFoundError(f'AF2FS parent seed {seed} SHA {expected} tidak ditemukan')
    if len(matches)>1:
        print(f'NOTE seed {seed}: {len(matches)} identical SHA copies; memakai pertama')
    return matches[0]

PARENTS={s:resolve_parent(s) for s in SEEDS}
for s in SEEDS:
    payload=json.loads(PARENT_RESULTS[s].read_text(encoding='utf-8'))
    if payload.get('test_images_accessed') is not False:
        raise RuntimeError(f'Parent result seed {s} tidak test-locked')
    print('AF2FS PARENT',s,':',PARENTS[s],sha256(PARENTS[s]))

OUT=WORK/'af2-igem-parent-confirmation-v1'
STATE_ZIP=WORK/'af2-igem-parent-confirmation-state.zip'
resume_states=input_matches(STATE_ZIP.name)
if len(resume_states)>1:
    raise RuntimeError(f'Resume state ambigu: {resume_states}')
if len(resume_states)==1 and not OUT.exists():
    print('RESTORE STATE:',resume_states[0])
    with zipfile.ZipFile(resume_states[0],'r') as z:
        z.extractall(WORK)
OUT.mkdir(parents=True,exist_ok=True)
print('OUTPUT:',OUT)


In [ ]:
from coffee_detector.af2_parent_residual.igem_confirmation import (
    AUDIT_REVISION,
    run_af2_igem_parent_static_audit,
)

STATIC={}
for s in SEEDS:
    path=OUT/'static_audits'/f'AF2FS_IGEM_static_seed{s}.json'
    path.parent.mkdir(parents=True,exist_ok=True)
    audit=None
    if path.is_file():
        cached=json.loads(path.read_text(encoding='utf-8'))
        if (
            cached.get('checkpoint_sha256')==sha256(PARENTS[s])
            and cached.get('audit_revision')==AUDIT_REVISION
            and cached.get('decision')=='PASS'
            and cached.get('training_authorized') is True
            and cached.get('test_access_authorized') is False
        ):
            audit=cached
            print('REUSE AUDIT:',s,AUDIT_REVISION)
    if audit is None:
        audit=run_af2_igem_parent_static_audit(PARENTS[s],path,device='cuda:0')

    failed={k:v for k,v in audit.get('gates',{}).items() if v is False and k!='test_accessed'}
    if (
        audit.get('decision')!='PASS'
        or audit.get('training_authorized') is not True
        or audit.get('test_access_authorized') is not False
        or audit.get('audit_revision')!=AUDIT_REVISION
        or failed
    ):
        raise RuntimeError(
            f'STOP static audit seed {s}; revision={audit.get("audit_revision")}; '
            f'failed_gates={failed}'
        )
    STATIC[s]=path
    r=audit['records']['AF2IGEM1']
    print(
        'STATIC PASS:',s,
        'revision=',AUDIT_REVISION,
        'trainable=',r['trainable_parameters'],'of',r['parameters'],
        'box_diff=',r['active_box_max_abs_diff'],
        'score_diff=',r['active_score_max_abs_diff'],
    )
print('ALL THREE STATIC AUDITS PASS')


In [ ]:
def snapshot_state():
    stage=WORK/'_igem_state_stage'
    if stage.exists():
        shutil.rmtree(stage)
    dst=stage/OUT.name
    dst.mkdir(parents=True)
    for name in ('val_reports','static_audits','logs'):
        src=OUT/name
        if src.exists():
            shutil.copytree(src,dst/name)
    for arm in ('AF2IGEM0','AF2IGEM1'):
        src_root=OUT/arm
        if not src_root.exists():
            continue
        for run_dir in src_root.iterdir():
            if not run_dir.is_dir():
                continue
            target=dst/arm/run_dir.name
            target.mkdir(parents=True,exist_ok=True)
            for filename in ('run_contract.json','training_complete.json','results.csv','args.yaml'):
                src=run_dir/filename
                if src.is_file():
                    shutil.copy2(src,target/filename)
    if STATE_ZIP.exists():
        STATE_ZIP.unlink()
    archive=Path(shutil.make_archive(
        str(STATE_ZIP.with_suffix('')),'zip',root_dir=stage,base_dir=OUT.name
    ))
    shutil.rmtree(stage)
    print('STATE SNAPSHOT:',archive,archive.stat().st_size,'bytes',flush=True)
    return archive

def result_path(arm,seed):
    return OUT/'val_reports'/f'{arm}_seed{seed}_result.json'

def run_arm(arm,seed):
    result=result_path(arm,seed)
    parent_sha=sha256(PARENTS[seed])
    parent_result_sha=sha256(PARENT_RESULTS[seed])
    if result.is_file():
        payload=json.loads(result.read_text(encoding='utf-8'))
        if (
            payload.get('format')=='coffee_detector.af2_parent_residual.igem_arm_result.v2'
            and payload.get('initial_af2_checkpoint_sha256')==parent_sha
            and payload.get('parent_result_sha256')==parent_result_sha
            and payload.get('static_audit_revision')==AUDIT_REVISION
            and payload.get('final_parent_state_bitwise_exact') is True
            and payload.get('test_images_accessed') is False
        ):
            print('REUSE COMPLETED:',arm,seed)
            return payload
        raise RuntimeError(f'Stale/mismatched result exists: {result}')

    log=OUT/'logs'/f'{arm}_seed{seed}.log'
    log.parent.mkdir(parents=True,exist_ok=True)
    cmd=[
        sys.executable,'-m','coffee_detector.experiments.run_faruq_v3_af2_igem_parent_arm',
        '--arm',arm,
        '--data-root',str(DATA),
        '--grouped-summary',str(GROUPED),
        '--af2-checkpoint',str(PARENTS[seed]),
        '--parent-result',str(PARENT_RESULTS[seed]),
        '--static-audit',str(STATIC[seed]),
        '--output-root',str(OUT),
        '--seed',str(seed),
        '--device','0',
        '--authorize-training',
    ]
    print('START/RESUME',arm,'seed',seed,'| log=',log,flush=True)
    with log.open('a',encoding='utf-8') as stream:
        p=subprocess.Popen(cmd,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)

    shown=-1
    while p.poll() is None:
        csv=OUT/arm/f'{arm}_seed{seed}'/'results.csv'
        epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epochs!=shown:
            print(f'{arm} seed {seed}: {epochs}/20 epoch tercatat',flush=True)
            shown=epochs
        time.sleep(120)

    if p.returncode:
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-200:]) if log.is_file() else '<no log>'
        raise RuntimeError(f'{arm} seed {seed} gagal rc={p.returncode}\n{tail}')
    if not result.is_file():
        raise FileNotFoundError(f'Result hilang setelah run: {result}')
    payload=json.loads(result.read_text(encoding='utf-8'))
    if payload.get('final_parent_state_bitwise_exact') is not True:
        raise RuntimeError(f'{arm} seed {seed}: serialized best.pt tidak mempertahankan parent exact')
    print('DONE:',arm,seed,{k:payload['metrics'][k] for k in (
        'macro_map50_95','bottom3_class_map50_95','worst_class_map50_95'
    )})
    snapshot_state()
    return payload

RESULTS={}
for seed in SEEDS:
    for arm in ('AF2IGEM0','AF2IGEM1'):
        RESULTS[(arm,seed)]=run_arm(arm,seed)
print('ALL SIX DEVELOPMENT RUNS COMPLETE')


In [ ]:
from coffee_detector.experiments.run_faruq_v3_af2_igem_parent_decision import run_igem_parent_decision

parents=[PARENT_RESULTS[s] for s in SEEDS]
controls=[result_path('AF2IGEM0',s) for s in SEEDS]
candidates=[result_path('AF2IGEM1',s) for s in SEEDS]
DECISION_PATH=OUT/'af2fs_igem_parent_confirmation_decision.json'
decision=run_igem_parent_decision(parents,controls,candidates,DECISION_PATH)

print('DECISION:',decision['decision'])
for metric,row in decision['aggregate'].items():
    print(
        metric,
        'parent=',round(100*row['parent_mean'],3),
        'control=',round(100*row['control_mean'],3),
        'candidate=',round(100*row['candidate_mean'],3),
        'cand-control pp=',round(100*row['candidate_minus_control_mean'],3),
        'cand-parent pp=',round(100*row['candidate_minus_parent_mean'],3),
    )
print('CRITERIA:',json.dumps(decision['criteria'],indent=2))
snapshot_state()

EVIDENCE=WORK/'af2-igem-parent-confirmation-evidence'
if EVIDENCE.exists():
    shutil.rmtree(EVIDENCE)
EVIDENCE.mkdir(parents=True)
for name in ('val_reports','static_audits','logs'):
    src=OUT/name
    if src.exists():
        shutil.copytree(src,EVIDENCE/name)
shutil.copy2(DECISION_PATH,EVIDENCE/DECISION_PATH.name)

parent_dir=EVIDENCE/'canonical_parents'
parent_dir.mkdir()
for s in SEEDS:
    shutil.copy2(PARENT_RESULTS[s],parent_dir/PARENT_RESULTS[s].name)

protocol_src=REPO/'docs/FARUQ_V3_AF2FS_IGEM_PARENT_CONFIRMATION_PROTOCOL_2026-08-24.md'
if protocol_src.is_file():
    shutil.copy2(protocol_src,EVIDENCE/protocol_src.name)
for arm in ('AF2IGEM0','AF2IGEM1'):
    shutil.copy2(REPO/f'configs/af2_parent_residual/{arm}.yaml',EVIDENCE/f'{arm}.yaml')

(EVIDENCE/'run_metadata.json').write_text(
    json.dumps({
        'branch':BRANCH,
        'commit':COMMIT,
        'audit_revision':AUDIT_REVISION,
        'seeds':SEEDS,
        'ultralytics':'8.4.96',
        'test_opened':False,
    },indent=2)+'\n',
    encoding='utf-8',
)
FINAL_ZIP=Path(shutil.make_archive(
    str(WORK/'af2-igem-parent-confirmation-output'),
    'zip',root_dir=WORK,base_dir=EVIDENCE.name
))
print('FINAL EVIDENCE ZIP:',FINAL_ZIP,FINAL_ZIP.stat().st_size,'bytes')
print('STATE ZIP:',STATE_ZIP,STATE_ZIP.stat().st_size if STATE_ZIP.exists() else 'missing')
print('TEST OPENED: FALSE')
